In [ ]:
from pathlib import Path

# ASTNN on the fixed CodeNet 4L non-clone-scope protocol.
# Attach codenet_4l_clean_data.zip, enable a T4-class GPU, and Run All.
DATASET_KEYS = ("codenet-4l-clean-data",)
RUN_LABEL = "astnn_clone_vs_mixed_aw_diff"


# Kaggle's current PyTorch build cannot execute kernels on Tesla P100 (sm_60).
# These notebooks are intended for a T4-class accelerator; two T4s are fine,
# although this single-process implementation uses GPU 0.
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU is enabled. In Kaggle select GPU accelerator: 2x T4.")
_GPU_CAPABILITY = torch.cuda.get_device_capability(0)
_GPU_NAME = torch.cuda.get_device_name(0)
if _GPU_CAPABILITY[0] < 7:
    raise RuntimeError(
        f"{_GPU_NAME} has unsupported CUDA capability sm_{_GPU_CAPABILITY[0]}{_GPU_CAPABILITY[1]}. "
        "Select 2x T4 in Kaggle Accelerator settings, restart the session, and Run All."
    )
print({"gpu": _GPU_NAME, "capability": f"sm_{_GPU_CAPABILITY[0]}{_GPU_CAPABILITY[1]}", "gpu_count": torch.cuda.device_count()})
# Runtime profile. Use quick_1h for preliminary results; change only this
# value to extended_6_7h for the larger follow-up run.
RUN_PROFILE = "final_full"
RUN_PRESETS = {"quick_1h": {'max_train_pairs': 50000, 'max_valid_pairs': 10000, 'max_test_pairs': 10000, 'epochs': 8, 'patience': 2}, "extended_6_7h": {'max_train_pairs': 100000, 'max_valid_pairs': 20000, 'max_test_pairs': 20000, 'epochs': 30, 'patience': 6}}
# Use this profile in every method notebook for a data-equal comparison.
RUN_PRESETS["comparison_50k"] = {
    **RUN_PRESETS["quick_1h"],
    "max_train_pairs": 50_000,
    "max_valid_pairs": 10_000,
    "max_test_pairs": 10_000,
}

# Final paper protocol: use every available pair in each official split.
RUN_PRESETS["final_full"] = {
    **RUN_PRESETS["quick_1h"],
    "max_train_pairs": None,
    "max_valid_pairs": None,
    "max_test_pairs": None,
}

# --- bounded run budget ---
# Kaggle sessions are capped, and a run that dies at the limit produces nothing.
# Training data stays large so results remain comparable with the published
# table; validation and test are capped because a bigger validation split only
# sharpens one threshold, and a bigger test split only tightens an error bar we
# do not report.
RUN_PRESETS["bounded_10h"] = {
    **RUN_PRESETS["comparison_50k"],
    "max_train_pairs": 200_000,
    "max_valid_pairs": 20_000,
    "max_test_pairs": 20_000,
}

if RUN_PROFILE not in RUN_PRESETS:
    raise ValueError(f"Unknown RUN_PROFILE={RUN_PROFILE!r}; choose one of {tuple(RUN_PRESETS)}")
RUN_CONFIG = RUN_PRESETS[RUN_PROFILE]


In [ ]:
# === per-language breakdown helper ===
# Splits an already-computed set of test predictions by the language of each
# pair. No retraining and no separate per-language model: this is the same run,
# reported per language so a strong average cannot hide a collapsed language.
import gzip as _gzip
import json as _json
from pathlib import Path as _Path

import numpy as _np
import pandas as _pd

_LANGUAGE_CACHE = {}
LANGUAGE_BREAKDOWN_ROWS = []


def _resolve_codes_file():
    for root in (_Path("/kaggle/input"), _Path("/kaggle/working"), _Path(".")):
        if not root.exists():
            continue
        for name in ("codes.jsonl.gz", "codes.jsonl", "codes.jsonl.gz.tmp"):
            for path in root.rglob(name):
                if path.is_file():
                    return path
    return None


def _open_any(path):
    with open(path, "rb") as probe:
        packed = probe.read(2) == b"\x1f\x8b"
    return _gzip.open(path, "rt", encoding="utf-8") if packed else open(path, "r", encoding="utf-8")


def code_languages():
    """``code_id -> language`` from the attached clean-data bundle."""
    if _LANGUAGE_CACHE:
        return _LANGUAGE_CACHE
    path = _resolve_codes_file()
    if path is None:
        print("[language-breakdown] codes.jsonl not found; breakdown will be skipped.")
        return _LANGUAGE_CACHE
    with _open_any(path) as stream:
        for line in stream:
            if not line.strip():
                continue
            record = _json.loads(line)
            code_id = str(record.get("code_id", record.get("id", record.get("idx", ""))))
            _LANGUAGE_CACHE[code_id] = str(record.get("language", record.get("lang", "unknown")))
    print(f"[language-breakdown] languages loaded for {len(_LANGUAGE_CACHE):,} codes.")
    return _LANGUAGE_CACHE


def _binary_scores(labels, predicted):
    labels = _np.asarray(labels, dtype=_np.int64)
    predicted = _np.asarray(predicted, dtype=_np.int64)
    tp = int(((predicted == 1) & (labels == 1)).sum())
    fp = int(((predicted == 1) & (labels == 0)).sum())
    tn = int(((predicted == 0) & (labels == 0)).sum())
    fn = int(((predicted == 0) & (labels == 1)).sum())
    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)
    f1 = 2 * precision * recall / max(1e-12, precision + recall)
    return {
        "P": precision, "R": recall, "F1": f1,
        "Acc": (tp + tn) / max(1, len(labels)),
        "TP": tp, "FP": fp, "TN": tn, "FN": fn,
        "Pairs": int(len(labels)), "Positives": int((labels == 1).sum()),
    }


def record_language_breakdown(frame, scores, threshold, *, dataset, method, graph_type=None):
    """Partition this run's test predictions by pair language and record them."""
    languages = code_languages()
    if not languages or frame is None or not len(frame):
        return []
    scores = _np.asarray(scores, dtype=_np.float64).reshape(-1)
    labels = _np.asarray(frame["label"], dtype=_np.int64).reshape(-1)
    if len(scores) != len(labels):
        print(f"[language-breakdown] skipped {method}: {len(scores)} scores vs {len(labels)} labels.")
        return []
    predicted = (scores >= float(threshold)).astype(_np.int64)

    left = [languages.get(str(value), "unknown") for value in frame["left_id"]]
    right = [languages.get(str(value), "unknown") for value in frame["right_id"]]
    # Cross-language pairs get their own bucket instead of being attributed to
    # one side; ATCoder is entirely java<->python and would otherwise vanish.
    keys = [a if a == b else f"{min(a, b)}->{max(a, b)}" for a, b in zip(left, right)]

    rows = []
    for key in sorted(set(keys)):
        mask = _np.asarray([value == key for value in keys])
        row = {"Dataset": dataset, "Method": method, "GraphType": graph_type or "", "Language": key}
        row.update(_binary_scores(labels[mask], predicted[mask]))
        row["Threshold"] = float(threshold)
        rows.append(row)
    overall = {"Dataset": dataset, "Method": method, "GraphType": graph_type or "", "Language": "ALL"}
    overall.update(_binary_scores(labels, predicted))
    overall["Threshold"] = float(threshold)
    rows.append(overall)

    LANGUAGE_BREAKDOWN_ROWS.extend(rows)
    table = _pd.DataFrame(LANGUAGE_BREAKDOWN_ROWS)
    out_path = _Path("/kaggle/working") / f"{dataset}_language_breakdown.csv"
    try:
        out_path.parent.mkdir(parents=True, exist_ok=True)
        table.to_csv(out_path, index=False)
    except OSError:
        out_path = _Path(f"{dataset}_language_breakdown.csv")
        table.to_csv(out_path, index=False)
    print(f"\n[language-breakdown] {method}{'/' + graph_type if graph_type else ''}")
    print(_pd.DataFrame(rows)[["Language", "P", "R", "F1", "Acc", "Pairs", "Positives"]].to_string(index=False))
    print(f"[language-breakdown] written to {out_path}")
    return rows

DATASET_KEY_FOR_BREAKDOWN = "atcoder_v3"


# ASTNN — CodeNet 4L — Clone vs mixed A/WA and different-problem non-clones

This baseline runs on 4,000 clone pairs against 2,000 same-problem Accepted/Wrong-Answer pairs and 2,000 different-problem pairs. It uses the same official splits, exact pair IDs, and language-uniform selection as the SPECTRA-Siam notebook for this scope.

DeepSim consumes CFG semantic matrices. For a code whose exported CFG is empty, the notebook uses that code's complete CPG adjacency as a declared fallback and records the affected code count. ASTNN and RtVNN consume the complete AST layer. Every baseline stops rather than silently dropping a selected pair.


In [ ]:
# Generated from research/faithful_graph_baselines; do not edit this cell by hand.


from collections.abc import Sequence

import numpy as np
import torch
from torch import Tensor, nn
import torch.nn.functional as F


def children_from_edges(num_nodes: int, rows: Sequence[int], cols: Sequence[int]) -> list[list[int]]:
    """Return a cycle-safe ordered child list for an exported AST."""
    children = [[] for _ in range(num_nodes)]
    seen: set[tuple[int, int]] = set()
    for parent, child in zip(rows, cols):
        parent, child = int(parent), int(child)
        edge = (parent, child)
        if 0 <= parent < num_nodes and 0 <= child < num_nodes and parent != child and edge not in seen:
            children[parent].append(child)
            seen.add(edge)
    return children


def postorder_depths(children: Sequence[Sequence[int]]) -> np.ndarray:
    """Compute bottom-up levels, breaking malformed back-edges safely."""
    n = len(children)
    state = np.zeros(n, dtype=np.int8)
    depth = np.zeros(n, dtype=np.int16)

    def visit(node: int) -> int:
        if state[node] == 2:
            return int(depth[node])
        if state[node] == 1:
            return 0
        state[node] = 1
        child_depths = [visit(c) for c in children[node] if state[c] != 1]
        depth[node] = 0 if not child_depths else min(255, 1 + max(child_depths))
        state[node] = 2
        return int(depth[node])

    for index in range(n):
        visit(index)
    return depth


def left_child_right_sibling(children: Sequence[Sequence[int]]) -> tuple[np.ndarray, np.ndarray]:
    """Convert an n-ary AST to the lossless left-child/right-sibling binary form."""
    n = len(children)
    left = np.full(n, -1, dtype=np.int64)
    right = np.full(n, -1, dtype=np.int64)
    for parent, child_list in enumerate(children):
        clean = [int(c) for c in child_list if 0 <= int(c) < n and int(c) != parent]
        if not clean:
            continue
        left[parent] = clean[0]
        for current, sibling in zip(clean, clean[1:]):
            right[current] = sibling
    return left, right


def deckard_subtree_vectors(
    node_types: Sequence[int],
    children: Sequence[Sequence[int]],
    vocab_size: int,
    *,
    min_nodes: int = 3,
) -> tuple[np.ndarray, np.ndarray]:
    """Construct DECKARD characteristic vectors for every eligible AST subtree.

    Each vector is a count of AST node categories in a rooted subtree.  The
    final coordinate stores subtree size, matching DECKARD's size-aware vector
    grouping while allowing direct scoring of benchmark-supplied pairs.
    """
    n = len(node_types)
    counts = np.zeros((n, vocab_size + 1), dtype=np.float32)
    sizes = np.ones(n, dtype=np.int32)
    levels = postorder_depths(children)
    for level in range(int(levels.max(initial=0)) + 1):
        for node in np.flatnonzero(levels == level):
            token = int(node_types[node])
            if 0 <= token < vocab_size:
                counts[node, token] += 1.0
            for child in children[node]:
                if 0 <= child < n and levels[child] < levels[node]:
                    counts[node] += counts[child]
                    sizes[node] += sizes[child]
            counts[node, -1] = float(sizes[node])
    keep = sizes >= min_nodes
    return counts[keep], sizes[keep]


def deckard_pair_similarity(
    left_vectors: np.ndarray,
    left_sizes: np.ndarray,
    right_vectors: np.ndarray,
    right_sizes: np.ndarray,
    *,
    size_tolerance: float = 0.20,
) -> float:
    """DECKARD-style best subtree similarity for one supplied code pair."""
    if not len(left_vectors) or not len(right_vectors):
        return 0.0
    best = 0.0
    # Pair evaluation replaces the original corpus-wide LSH candidate lookup;
    # the characteristic vectors and size filter remain unchanged.
    for i, size in enumerate(left_sizes):
        ratio = np.abs(right_sizes.astype(np.float32) - float(size)) / max(float(size), 1.0)
        candidates = np.flatnonzero(ratio <= size_tolerance)
        if not len(candidates):
            continue
        a = left_vectors[i]
        b = right_vectors[candidates]
        denom = np.linalg.norm(b, axis=1) * max(float(np.linalg.norm(a)), 1e-8)
        sims = (b @ a) / np.maximum(denom, 1e-8)
        best = max(best, float(sims.max(initial=0.0)))
    return best


def _masked_max(values: Tensor, mask: Tensor, dim: int) -> Tensor:
    masked = values.masked_fill(~mask.unsqueeze(-1), torch.finfo(values.dtype).min)
    result = masked.max(dim=dim).values
    return torch.where(torch.isfinite(result), result, torch.zeros_like(result))


class RecursiveTreeEncoder(nn.Module):
    """Bottom-up recursive tree encoder over an exported AST."""

    def __init__(self, vocab_size: int, embed_dim: int, hidden_dim: int):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.node = nn.Linear(embed_dim, hidden_dim)
        self.child = nn.Linear(hidden_dim, hidden_dim, bias=False)

    def forward(self, node_types: Tensor, parents: Tensor, children: Tensor, depths: Tensor) -> Tensor:
        mask = node_types.ne(0)
        embedded = self.embedding(node_types)
        hidden = torch.zeros((*node_types.shape, self.node.out_features), device=node_types.device, dtype=embedded.dtype)
        valid_edges = parents.ge(0) & children.ge(0)
        max_depth = int(depths.max().detach().cpu()) if depths.numel() else 0
        for level in range(max_depth + 1):
            aggregate = torch.zeros_like(hidden)
            valid = valid_edges & depths.gather(1, children.clamp_min(0)).lt(level + 1)
            child_index = children.clamp_min(0).unsqueeze(-1).expand(-1, -1, hidden.size(-1))
            child_values = hidden.gather(1, child_index) * valid.unsqueeze(-1)
            parent_index = parents.clamp_min(0).unsqueeze(-1).expand_as(child_values)
            aggregate.scatter_add_(1, parent_index, child_values)
            count = torch.zeros_like(hidden[..., :1])
            count.scatter_add_(1, parents.clamp_min(0).unsqueeze(-1), valid.unsqueeze(-1).to(hidden.dtype))
            candidate = torch.tanh(self.node(embedded) + self.child(aggregate / count.clamp_min(1.0)))
            update = depths.eq(level) & mask
            hidden = torch.where(update.unsqueeze(-1), candidate, hidden)
        return hidden * mask.unsqueeze(-1)


class ASTNNEncoder(nn.Module):
    """ASTNN block sequence: recursive subtree encoding, BiGRU, max pooling."""

    def __init__(self, vocab_size: int, embed_dim: int = 128, hidden_dim: int = 100, code_dim: int = 200):
        super().__init__()
        self.tree = RecursiveTreeEncoder(vocab_size, embed_dim, hidden_dim)
        self.gru = nn.GRU(hidden_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.projection = nn.Linear(hidden_dim * 2, code_dim)

    def forward(self, node_types: Tensor, parents: Tensor, children: Tensor, depths: Tensor, statements: Tensor) -> Tensor:
        nodes = self.tree(node_types, parents, children, depths)
        statement_mask = statements.ge(0)
        indices = statements.clamp_min(0).unsqueeze(-1).expand(-1, -1, nodes.size(-1))
        blocks = nodes.gather(1, indices) * statement_mask.unsqueeze(-1)
        sequence, _ = self.gru(blocks)
        pooled = _masked_max(sequence, statement_mask, 1)
        return torch.tanh(self.projection(pooled))


class RtvNNEncoder(nn.Module):
    """Recursive tree/vector encoder with a reconstruction objective."""

    def __init__(self, vocab_size: int, embed_dim: int = 128, hidden_dim: int = 192, code_dim: int = 192):
        super().__init__()
        self.tree = RecursiveTreeEncoder(vocab_size, embed_dim, hidden_dim)
        self.decoder = nn.Linear(hidden_dim, embed_dim)
        self.output = nn.Linear(hidden_dim * 2, code_dim)

    def forward(self, node_types: Tensor, parents: Tensor, children: Tensor, depths: Tensor) -> tuple[Tensor, Tensor]:
        nodes = self.tree(node_types, parents, children, depths)
        mask = node_types.ne(0)
        mean = (nodes * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True).clamp_min(1)
        maximum = _masked_max(nodes, mask, 1)
        code = torch.tanh(self.output(torch.cat([mean, maximum], dim=-1)))
        target = self.tree.embedding(node_types).detach()
        reconstruction = F.mse_loss(self.decoder(nodes)[mask], target[mask]) if mask.any() else nodes.sum() * 0
        return code, reconstruction


class BinaryTreeLSTMEncoder(nn.Module):
    """Binary Tree-LSTM over left-child/right-sibling AST conversion."""

    def __init__(self, vocab_size: int, embed_dim: int = 128, hidden_dim: int = 192):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gates = nn.Linear(embed_dim + hidden_dim * 2, hidden_dim * 5)
        self.hidden_dim = hidden_dim

    def forward(self, node_types: Tensor, left: Tensor, right: Tensor, depths: Tensor) -> tuple[Tensor, Tensor]:
        embedded = self.embedding(node_types)
        batch, nodes = node_types.shape
        h = embedded.new_zeros(batch, nodes, self.hidden_dim)
        c = embedded.new_zeros(batch, nodes, self.hidden_dim)
        max_depth = int(depths.max().detach().cpu()) if depths.numel() else 0
        for level in range(max_depth + 1):
            li, ri = left.clamp_min(0), right.clamp_min(0)
            gather = lambda value, index: value.gather(1, index.unsqueeze(-1).expand(-1, -1, value.size(-1)))
            hl, hr = gather(h, li) * left.ge(0).unsqueeze(-1), gather(h, ri) * right.ge(0).unsqueeze(-1)
            cl, cr = gather(c, li) * left.ge(0).unsqueeze(-1), gather(c, ri) * right.ge(0).unsqueeze(-1)
            i, fl, fr, o, u = self.gates(torch.cat([embedded, hl, hr], -1)).chunk(5, -1)
            new_c = torch.sigmoid(i) * torch.tanh(u) + torch.sigmoid(fl) * cl + torch.sigmoid(fr) * cr
            new_h = torch.sigmoid(o) * torch.tanh(new_c)
            update = depths.eq(level) & node_types.ne(0)
            h = torch.where(update.unsqueeze(-1), new_h, h)
            c = torch.where(update.unsqueeze(-1), new_c, c)
        return h, c


class CDLHModel(nn.Module):
    """Binary Tree-LSTM followed by a supervised continuous hash layer."""

    def __init__(self, vocab_size: int, embed_dim: int = 128, hidden_dim: int = 192, hash_bits: int = 32):
        super().__init__()
        self.tree_lstm = BinaryTreeLSTMEncoder(vocab_size, embed_dim, hidden_dim)
        self.hash = nn.Linear(hidden_dim, hash_bits)

    def encode(self, node_types: Tensor, left: Tensor, right: Tensor, depths: Tensor) -> Tensor:
        hidden, _ = self.tree_lstm(node_types, left, right, depths)
        mask = node_types.ne(0)
        return torch.tanh(self.hash(_masked_max(hidden, mask, 1)))

    @staticmethod
    def loss(left_hash: Tensor, right_hash: Tensor, labels: Tensor, quantization_weight: float = 0.01) -> Tensor:
        similarity = (left_hash * right_hash).mean(-1)
        pair_loss = F.binary_cross_entropy_with_logits(4.0 * similarity, labels.float())
        quantization = ((left_hash.abs() - 1.0) ** 2).mean() + ((right_hash.abs() - 1.0) ** 2).mean()
        return pair_loss + quantization_weight * quantization


def relation_message(hidden: Tensor, edge_src: Tensor, edge_dst: Tensor, transform: nn.Module) -> Tensor:
    """Batched sparse relation aggregation from padded edge-index tensors."""
    valid = edge_src.ge(0) & edge_dst.ge(0)
    src = edge_src.clamp_min(0)
    dst = edge_dst.clamp_min(0)
    src_hidden = hidden.gather(1, src.unsqueeze(-1).expand(-1, -1, hidden.size(-1)))
    messages = transform(src_hidden) * valid.unsqueeze(-1)
    output = torch.zeros_like(hidden)
    output.scatter_add_(1, dst.unsqueeze(-1).expand_as(messages), messages)
    counts = torch.zeros_like(hidden[..., :1])
    counts.scatter_add_(1, dst.unsqueeze(-1), valid.unsqueeze(-1).to(hidden.dtype))
    return output / counts.clamp_min(1.0)


class DeepSimEncoder(nn.Module):
    """CFG semantic-matrix encoder augmented with aligned DDG features."""

    def __init__(self, vocab_size: int, numeric_dim: int = 6, embed_dim: int = 96, hidden_dim: int = 192, code_dim: int = 192):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.self_proj = nn.Linear(embed_dim + numeric_dim, hidden_dim)
        self.cfg_message = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.sequence = nn.GRU(hidden_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.output = nn.Linear(hidden_dim * 2, code_dim)

    def forward(self, node_types: Tensor, numeric: Tensor, cfg_src: Tensor, cfg_dst: Tensor) -> Tensor:
        mask = node_types.ne(0)
        semantic_matrix = torch.cat([self.embedding(node_types), numeric], -1)
        semantic_hidden = self.self_proj(semantic_matrix)
        hidden = torch.tanh(semantic_hidden + relation_message(semantic_hidden, cfg_src, cfg_dst, self.cfg_message))
        sequence, _ = self.sequence(hidden)
        return F.normalize(self.output(_masked_max(sequence, mask, 1)), dim=-1)


class FAASTGGNN(nn.Module):
    """Relation-aware GGNN on our AST+CFG+DDG flow-augmented graph."""

    def __init__(self, vocab_size: int, relations: int = 3, hidden_dim: int = 192, code_dim: int = 192, steps: int = 5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_dim, padding_idx=0)
        self.transforms = nn.ModuleList(nn.Linear(hidden_dim, hidden_dim, bias=False) for _ in range(relations))
        self.gru = nn.GRUCell(hidden_dim, hidden_dim)
        self.gate = nn.Linear(hidden_dim, 1)
        self.output = nn.Linear(hidden_dim, code_dim)
        self.steps = steps

    def propagate(self, hidden: Tensor, edge_src: Tensor, edge_dst: Tensor) -> Tensor:
        aggregate = torch.zeros_like(hidden)
        for relation, transform in enumerate(self.transforms):
            aggregate += relation_message(hidden, edge_src[:, relation], edge_dst[:, relation], transform)
        return self.gru(aggregate.flatten(0, 1), hidden.flatten(0, 1)).view_as(hidden)

    def pool(self, hidden: Tensor, mask: Tensor) -> Tensor:
        weights = torch.sigmoid(self.gate(hidden)) * mask.unsqueeze(-1)
        pooled = (weights * hidden).sum(1) / weights.sum(1).clamp_min(1e-6)
        return F.normalize(self.output(pooled), dim=-1)

    def forward(self, node_types: Tensor, edge_src: Tensor, edge_dst: Tensor) -> Tensor:
        mask = node_types.ne(0)
        hidden = self.embedding(node_types)
        for _ in range(self.steps):
            hidden = self.propagate(hidden, edge_src, edge_dst) * mask.unsqueeze(-1)
        return self.pool(hidden, mask)


class FAASTGMN(FAASTGGNN):
    """FA-AST GGNN with pair-aware cross-graph matching at every step."""

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        hidden_dim = self.embedding.embedding_dim
        self.match_gru = nn.GRUCell(hidden_dim * 2, hidden_dim)

    @staticmethod
    def cross_attention(left: Tensor, right: Tensor, left_mask: Tensor, right_mask: Tensor) -> tuple[Tensor, Tensor]:
        scale = left.size(-1) ** -0.5
        scores = torch.bmm(left, right.transpose(1, 2)) * scale
        left_scores = scores.masked_fill(~right_mask.unsqueeze(1), -1e4)
        right_scores = scores.transpose(1, 2).masked_fill(~left_mask.unsqueeze(1), -1e4)
        left_match = torch.bmm(torch.softmax(left_scores, -1), right)
        right_match = torch.bmm(torch.softmax(right_scores, -1), left)
        return left_match, right_match

    def forward_pair(
        self,
        left_types: Tensor,
        left_src: Tensor,
        left_dst: Tensor,
        right_types: Tensor,
        right_src: Tensor,
        right_dst: Tensor,
    ) -> tuple[Tensor, Tensor]:
        lm, rm = left_types.ne(0), right_types.ne(0)
        left, right = self.embedding(left_types), self.embedding(right_types)
        for _ in range(self.steps):
            lp, rp = self.propagate(left, left_src, left_dst), self.propagate(right, right_src, right_dst)
            lx, rx = self.cross_attention(lp, rp, lm, rm)
            left = self.match_gru(torch.cat([lp, lp - lx], -1).flatten(0, 1), left.flatten(0, 1)).view_as(left)
            right = self.match_gru(torch.cat([rp, rp - rx], -1).flatten(0, 1), right.flatten(0, 1)).view_as(right)
            left, right = left * lm.unsqueeze(-1), right * rm.unsqueeze(-1)
        return self.pool(left, lm), self.pool(right, rm)



import copy
import datetime
import gzip
import shutil
import zipfile
import json
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, precision_recall_fscore_support, roc_auc_score
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

try:
    children_from_edges
except NameError:  # Direct local import; generated notebooks prepend core.py.
    from research.faithful_graph_baselines.core import (
        ASTNNEncoder,
        CDLHModel,
        DeepSimEncoder,
        FAASTGGNN,
        FAASTGMN,
        RtvNNEncoder,
        children_from_edges,
        deckard_pair_similarity,
        deckard_subtree_vectors,
        left_child_right_sibling,
        postorder_depths,
    )


METHOD_CONFIGS = {
    "deckard": {"name": "Deckard", "batch": None, "lr": None, "weight_decay": None},
    "astnn": {"name": "ASTNN", "batch": 128, "lr": 1e-3, "weight_decay": 0.0},
    "rtvnn": {"name": "RtvNN", "batch": 256, "lr": 5e-4, "weight_decay": 1e-4},
    "cdlh": {"name": "CDLH", "batch": 128, "lr": 1e-3, "weight_decay": 0.0},
    "deepsim": {"name": "DeepSim", "batch": 128, "lr": 5e-4, "weight_decay": 1e-3},
    "fa_ast_ggnn": {"name": "FA-AST+GGNN", "batch": 64, "lr": 5e-4, "weight_decay": 1e-4},
    "fa_ast_gmn": {"name": "FA-AST+GMN", "batch": 32, "lr": 5e-4, "weight_decay": 1e-4},
}

MAX_NODES = 256
MAX_EDGES = 512
MAX_STATEMENTS = 64
RELATIONS = ("ast", "cfg", "ddg")


def _seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def _open_text(path: Path):
    with path.open("rb") as probe:
        packed = probe.read(2) == b"\x1f\x8b"
    return gzip.open(path, "rt", encoding="utf-8") if packed else path.open("r", encoding="utf-8")


_CODENET_INPUT_FILES = ("pairs.csv.gz", "graph_spectra.jsonl.gz", "codes.jsonl.gz", "metadata.json")
_CODENET_EXTRACT_ROOT = Path("/kaggle/working/codenet_4l_clean_data_input")


def _complete_codenet_bundle(root: Path) -> Path | None:
    """Find one directory containing both pair rows and graph spectra."""
    if not root.exists():
        return None
    # Kaggle may unpack nested gzip members asymmetrically: pairs.csv.gz can
    # become pairs.csv while graph_spectra.jsonl.gz becomes
    # graph_spectra.jsonl/graph_spectra.jsonl.gz.tmp. Treat that directory tree
    # as one bundle instead of requiring both files to be direct siblings.
    pair_paths = sorted(path for path in root.rglob("pairs.csv*") if path.is_file())
    for pair_path in pair_paths:
        bundle_root = pair_path.parent
        graph_paths = [
            path for path in bundle_root.rglob("graph_spectra.jsonl*")
            if path.is_file()
        ]
        if graph_paths:
            return bundle_root
    return None


def _materialize_codenet_zip_input() -> Path:
    """Return a complete CodeNet bundle, extracting its ZIP when necessary."""
    input_root = Path("/kaggle/input")
    existing = _complete_codenet_bundle(input_root)
    if existing is not None:
        print(f"[input] using complete CodeNet bundle: {existing}")
        return existing

    exact = sorted(input_root.rglob("codenet_4l_clean_data.zip")) if input_root.exists() else []
    all_zips = sorted(input_root.rglob("*.zip")) if input_root.exists() else []
    candidates = exact + [path for path in all_zips if path not in set(exact)]
    inspected = []
    for candidate in candidates:
        try:
            with zipfile.ZipFile(candidate) as archive:
                members = {Path(name).name: name for name in archive.namelist() if not name.endswith("/")}
                inspected.append({"zip": str(candidate), "members": sorted(set(_CODENET_INPUT_FILES) & set(members))})
                if not all(name in members for name in _CODENET_INPUT_FILES):
                    continue
                _CODENET_EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
                for name in _CODENET_INPUT_FILES:
                    target = _CODENET_EXTRACT_ROOT / name
                    with archive.open(members[name]) as src, target.open("wb") as dst:
                        shutil.copyfileobj(src, dst)
                print(f"[input] extracted complete CodeNet bundle: {candidate} -> {_CODENET_EXTRACT_ROOT}")
                return _CODENET_EXTRACT_ROOT
        except zipfile.BadZipFile:
            inspected.append({"zip": str(candidate), "error": "bad_zip"})

    visible_pairs = sorted(str(path) for path in input_root.rglob("pairs.csv*") if path.is_file()) if input_root.exists() else []
    visible_graphs = sorted(str(path) for path in input_root.rglob("graph_spectra.jsonl*") if path.is_file()) if input_root.exists() else []
    raise FileNotFoundError(
        "No complete CodeNet clean-data bundle was found. Attach codenet_4l_clean_data.zip as a Kaggle input. "
        f"Visible pairs={visible_pairs[:5]}, visible graphs={visible_graphs[:5]}, inspected ZIPs={inspected[:5]}"
    )


def _bundle_file(bundle_root: Path, *names: str) -> Path:
    for name in names:
        path = bundle_root / name
        if path.is_file():
            return path
    for name in names:
        matches = sorted(path for path in bundle_root.rglob(name) if path.is_file())
        if matches:
            return matches[0]
    raise FileNotFoundError(f"CodeNet bundle {bundle_root} is missing all of {names}")


def _find_input(dataset_key: str, *names: str) -> Path:
    roots = [Path("/kaggle/input") / dataset_key, Path("/kaggle/input"), Path(".")]
    matches = []
    for root in roots:
        if not root.exists():
            continue
        for name in names:
            matches.extend(path for path in root.rglob(name) if path.is_file())
    if not matches:
        raise FileNotFoundError(f"Could not find {names} below the Kaggle inputs")
    return sorted(matches, key=lambda path: (len(path.parts), len(path.name), str(path)))[0]


def _metric_dict(labels, scores, threshold: float) -> dict:
    labels = np.asarray(labels, dtype=np.int64)
    predictions = (np.asarray(scores) >= threshold).astype(np.int64)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="binary", zero_division=0
    )
    return {
        "P": float(precision),
        "R": float(recall),
        "F1": float(f1),
        "Acc": float(accuracy_score(labels, predictions)),
        "MacroF1": float(f1_score(labels, predictions, average="macro", zero_division=0)),
        "BalancedAccuracy": float(balanced_accuracy_score(labels, predictions)),
        "ROC_AUC": float(roc_auc_score(labels, scores)) if len(np.unique(labels)) == 2 else float("nan"),
        "TP": int(((labels == 1) & (predictions == 1)).sum()),
        "FP": int(((labels == 0) & (predictions == 1)).sum()),
        "TN": int(((labels == 0) & (predictions == 0)).sum()),
        "FN": int(((labels == 1) & (predictions == 0)).sum()),
    }


def _choose_threshold(labels, scores) -> tuple[float, dict]:
    scores = np.asarray(scores, dtype=np.float32)
    candidates = np.unique(
        np.concatenate([np.linspace(0.01, 0.99, 199), np.quantile(scores, np.linspace(0, 1, 201))])
    )
    best_threshold, best_metrics = 0.5, None
    for threshold in candidates:
        metrics = _metric_dict(labels, scores, float(threshold))
        key = (metrics["F1"], metrics["BalancedAccuracy"])
        if best_metrics is None or key > (best_metrics["F1"], best_metrics["BalancedAccuracy"]):
            best_threshold, best_metrics = float(threshold), metrics
    return best_threshold, best_metrics


EXPECTED_CONFIGURATIONS = ('python', 'java', 'cpp', 'csharp', 'python_java', 'python_cpp', 'python_csharp', 'java_cpp', 'java_csharp', 'cpp_csharp')
PAIR_KIND_TARGETS_PER_CONFIGURATION = {'train': {'clone': 280, 'hard_nonclone': 140, 'nonclone_diff_problem': 140}, 'valid': {'clone': 60, 'hard_nonclone': 30, 'nonclone_diff_problem': 30}, 'test': {'clone': 60, 'hard_nonclone': 30, 'nonclone_diff_problem': 30}}
NONCLONE_KINDS = ('hard_nonclone', 'nonclone_diff_problem')
FORBIDDEN_PAIR_KINDS = ("nonclone_mutation",)
PAIR_SCOPE_AUDIT = {}


def select_nonclone_scope_protocol(frame: pd.DataFrame, split: str) -> tuple[pd.DataFrame, dict]:
    """Select exact, language-uniform buckets for this fixed experiment."""
    required = {"pair_kind", "configuration_id", "label"}
    missing = sorted(required - set(frame.columns))
    if missing:
        raise RuntimeError(f"{split} is missing required CodeNet provenance columns: {missing}")
    if split not in PAIR_KIND_TARGETS_PER_CONFIGURATION:
        raise ValueError(f"Unexpected split: {split!r}")

    working = frame.copy()
    working["pair_kind"] = working["pair_kind"].fillna("").astype(str)
    working["configuration_id"] = working["configuration_id"].fillna("").astype(str)
    working["label"] = working["label"].astype(int)
    selected_parts = []
    selected_buckets = {}
    targets = PAIR_KIND_TARGETS_PER_CONFIGURATION[split]
    sort_columns = [
        column for column in ("sample_rank", "pair_id", "left_id", "right_id")
        if column in working.columns
    ]

    for configuration in EXPECTED_CONFIGURATIONS:
        for pair_kind, target in targets.items():
            expected_label = 1 if pair_kind == "clone" else 0
            bucket = working[
                working.configuration_id.eq(configuration)
                & working.pair_kind.eq(pair_kind)
                & working.label.eq(expected_label)
            ].copy()
            if len(bucket) < target:
                raise RuntimeError(
                    f"{split}/{configuration}/{pair_kind} has {len(bucket)} pairs; "
                    f"the fixed protocol requires {target}. Attach the 12k scope-study clean dataset."
                )
            if sort_columns:
                bucket = bucket.sort_values(sort_columns, kind="stable")
            selected_parts.append(bucket.head(target))
            selected_buckets[f"{configuration}/{pair_kind}"] = int(target)

    selected = pd.concat(selected_parts, ignore_index=True)
    if selected.pair_kind.isin(FORBIDDEN_PAIR_KINDS).any():
        raise RuntimeError("Mutation-derived pairs are forbidden in the current non-clone scope study.")
    if set(selected.label.unique()) != {0, 1}:
        raise RuntimeError(f"{split} does not contain both classes after fixed-scope selection.")
    if sort_columns:
        selected = selected.sort_values(
            ["configuration_id", "pair_kind", *sort_columns], kind="stable"
        ).reset_index(drop=True)

    audit = {
        "split": split,
        "input_pairs": int(len(frame)),
        "input_by_pair_kind": {
            str(kind): int(count)
            for kind, count in working.pair_kind.value_counts().sort_index().items()
        },
        "target_per_configuration": targets,
        "selected_buckets": selected_buckets,
        "selected_pairs_before_graph_join": int(len(selected)),
        "selected_by_pair_kind": {
            str(kind): int(count)
            for kind, count in selected.pair_kind.value_counts().sort_index().items()
        },
        "selected_labels": {
            str(label): int(count)
            for label, count in selected.label.value_counts().sort_index().items()
        },
        "excluded_mutation_pairs": int(working.pair_kind.isin(FORBIDDEN_PAIR_KINDS).sum()),
        "pair_cap": "disabled_for_fixed_protocol",
    }
    return selected, audit


def _limit(frame: pd.DataFrame, split: str, maximum: int | None, seed: int) -> pd.DataFrame:
    subset = frame[frame.split == split].copy()
    if maximum is not None and len(subset) > maximum:
        subset = subset.sample(maximum, random_state=seed)
    return subset.reset_index(drop=True)


def _adjacency(layer: dict) -> dict:
    return layer.get("adjacency", {}) if isinstance(layer, dict) else {}


def _load_raw_graphs(path: Path, wanted: set[str]) -> dict[str, dict]:
    graphs = {}
    with _open_text(path) as source:
        for line in tqdm(source, desc="Loading exported program graphs", unit="code"):
            if not line.strip():
                continue
            row = json.loads(line)
            code_id = str(row.get("code_id"))
            if code_id in wanted:
                graphs[code_id] = {key: _adjacency(value) for key, value in row.get("graphs", {}).items()}
    return graphs


def _training_type_vocabulary(raw: dict[str, dict], train_ids: set[str], graph_view: str) -> dict[str, int]:
    frequency: dict[str, int] = {}
    for code_id in train_ids:
        adjacency = raw.get(code_id, {}).get(graph_view, {})
        for node_type in adjacency.get("node_types", [])[:MAX_NODES]:
            value = str(node_type)
            frequency[value] = frequency.get(value, 0) + 1
    ordered = sorted(frequency.items(), key=lambda item: (-item[1], item[0]))
    return {"<pad>": 0, "<unk>": 1, **{value: index + 2 for index, (value, _) in enumerate(ordered)}}


def _statement_roots(types: list[str], children: list[list[int]]) -> list[int]:
    indegree = [0] * len(types)
    for child_list in children:
        for child in child_list:
            indegree[child] += 1
    roots = [index for index, degree in enumerate(indegree) if degree == 0]
    methods = [index for index in roots if types[index].upper() == "METHOD"] or roots[:1]
    statements = []
    for method in methods:
        direct = children[method]
        blocks = [node for node in direct if types[node].upper() == "BLOCK"]
        for parent in blocks or [method]:
            statements.extend(children[parent])
    return list(dict.fromkeys(statements or methods or [0]))[:MAX_STATEMENTS]


def _tree_arrays(adjacency: dict, vocab: dict[str, int]):
    raw_types = [str(value) for value in adjacency.get("node_types", [])]
    n = min(int(adjacency.get("num_nodes", 0) or 0), len(raw_types), MAX_NODES)
    rows, cols = [], []
    for parent, child in zip(adjacency.get("row", []), adjacency.get("col", [])):
        parent, child = int(parent), int(child)
        if parent < n and child < n and len(rows) < MAX_EDGES:
            rows.append(parent)
            cols.append(child)
    children = children_from_edges(n, rows, cols)
    depth = postorder_depths(children)
    left, right = left_child_right_sibling(children)
    binary_children = [list(filter(lambda value: value >= 0, (int(left[index]), int(right[index])))) for index in range(n)]
    binary_depth = postorder_depths(binary_children)
    types = np.zeros(MAX_NODES, dtype=np.int32)
    types[:n] = [vocab.get(value, 1) for value in raw_types[:n]]
    parents = np.full(MAX_EDGES, -1, dtype=np.int16)
    child_array = np.full(MAX_EDGES, -1, dtype=np.int16)
    parents[:len(rows)], child_array[:len(cols)] = rows, cols
    depths = np.zeros(MAX_NODES, dtype=np.int16)
    binary_depths = np.zeros(MAX_NODES, dtype=np.int16)
    left_array = np.full(MAX_NODES, -1, dtype=np.int16)
    right_array = np.full(MAX_NODES, -1, dtype=np.int16)
    depths[:n], binary_depths[:n], left_array[:n], right_array[:n] = depth, binary_depth, left, right
    statements = np.full(MAX_STATEMENTS, -1, dtype=np.int16)
    roots = _statement_roots(raw_types[:n], children) if n else []
    statements[:len(roots)] = roots
    return types, parents, child_array, depths, binary_depths, left_array, right_array, statements, children


def _map_relation(layer: dict, universe_ids: list[str]) -> tuple[np.ndarray, np.ndarray]:
    lookup = {str(node_id): index for index, node_id in enumerate(universe_ids)}
    layer_ids = [str(value) for value in layer.get("node_ids", [])]
    source = np.full(MAX_EDGES, -1, dtype=np.int16)
    target = np.full(MAX_EDGES, -1, dtype=np.int16)
    edges = []
    for row, col in zip(layer.get("row", []), layer.get("col", [])):
        row, col = int(row), int(col)
        if row < len(layer_ids) and col < len(layer_ids):
            u, v = lookup.get(layer_ids[row]), lookup.get(layer_ids[col])
            if u is not None and v is not None and u < MAX_NODES and v < MAX_NODES:
                edges.append((u, v))
                if len(edges) >= MAX_EDGES:
                    break
    if edges:
        source[:len(edges)], target[:len(edges)] = zip(*edges)
    return source, target


def _flow_arrays(graphs: dict, vocab: dict[str, int]):
    cpg = graphs.get("cpg") or graphs.get("ast") or {}
    universe_ids = [str(value) for value in cpg.get("node_ids", [])][:MAX_NODES]
    universe_types = [str(value) for value in cpg.get("node_types", [])][:len(universe_ids)]
    types = np.zeros(MAX_NODES, dtype=np.int32)
    types[:len(universe_types)] = [vocab.get(value, 1) for value in universe_types]
    sources = np.full((len(RELATIONS), MAX_EDGES), -1, dtype=np.int16)
    targets = np.full_like(sources, -1)
    for relation, name in enumerate(RELATIONS):
        sources[relation], targets[relation] = _map_relation(graphs.get(name, {}), universe_ids)
    return types, sources, targets


def _deepsim_arrays(graphs: dict, vocab: dict[str, int]):
    cfg = graphs.get("cfg", {})
    cfg_ids = [str(value) for value in cfg.get("node_ids", [])][:MAX_NODES]
    cfg_types = [str(value) for value in cfg.get("node_types", [])][:len(cfg_ids)]
    types = np.zeros(MAX_NODES, dtype=np.int32)
    types[:len(cfg_types)] = [vocab.get(value, 1) for value in cfg_types]
    src, dst = _map_relation(cfg, cfg_ids)
    numeric = np.zeros((MAX_NODES, 6), dtype=np.float32)
    lookup = {node_id: index for index, node_id in enumerate(cfg_ids)}
    for u, v in zip(src[src >= 0], dst[dst >= 0]):
        numeric[int(u), 0] += 1
        numeric[int(v), 1] += 1
    numeric[:, 2] = numeric[:, 0] + numeric[:, 1]
    numeric[:, 3] = (numeric[:, 0] > 1).astype(np.float32)
    numeric[:, 4] = (numeric[:, 2] > 0).astype(np.float32)
    if cfg_ids:
        numeric[:len(cfg_ids), 5] = np.linspace(0, 1, len(cfg_ids), dtype=np.float32)
    numeric[:, :3] = np.log1p(numeric[:, :3])
    return types, numeric, src, dst


class _PairRows(Dataset):
    def __init__(self, frame: pd.DataFrame, id_to_row: dict[str, int]):
        self.left = frame.left_id.map(id_to_row).to_numpy(np.int64)
        self.right = frame.right_id.map(id_to_row).to_numpy(np.int64)
        self.labels = frame.label.to_numpy(np.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.left[index], self.right[index], self.labels[index]


class _PairModel(nn.Module):
    def __init__(self, method: str, arrays: dict[str, np.ndarray], vocab_size: int):
        super().__init__()
        self.method = method
        for name, value in arrays.items():
            self.register_buffer(name, torch.from_numpy(value), persistent=False)
        if method == "astnn":
            self.encoder = ASTNNEncoder(vocab_size)
            code_dim = 200
        elif method == "rtvnn":
            self.encoder = RtvNNEncoder(vocab_size)
            code_dim = 192
        elif method == "cdlh":
            self.encoder = CDLHModel(vocab_size)
            code_dim = 64
        elif method == "deepsim":
            self.encoder = DeepSimEncoder(vocab_size)
            code_dim = 192
        elif method == "fa_ast_ggnn":
            self.encoder = FAASTGGNN(vocab_size)
            code_dim = 192
        elif method == "fa_ast_gmn":
            self.encoder = FAASTGMN(vocab_size)
            code_dim = 192
        else:
            raise ValueError(method)
        if method == "astnn":
            # The official clone model applies one symmetric linear decision
            # layer to the absolute difference between code vectors.
            self.classifier = nn.Linear(code_dim, 1)
        elif method != "cdlh":
            self.classifier = nn.Sequential(
                nn.Linear(code_dim * 2 + 2, 192), nn.ReLU(), nn.Dropout(0.2), nn.Linear(192, 1)
            )

    def _tree(self, rows: Tensor):
        args = (
            self.node_types[rows].long(), self.edge_parents[rows].long(),
            self.edge_children[rows].long(), self.depths[rows].long()
        )
        if self.method == "astnn":
            return self.encoder(*args, self.statements[rows].long()), args[0].new_zeros((), dtype=torch.float32)
        if self.method == "rtvnn":
            return self.encoder(*args)
        return self.encoder.encode(
            args[0], self.left_child[rows].long(), self.right_sibling[rows].long(),
            self.binary_depths[rows].long()
        ), args[0].new_zeros((), dtype=torch.float32)

    def _single(self, rows: Tensor):
        if self.method in {"astnn", "rtvnn", "cdlh"}:
            return self._tree(rows)
        if self.method == "deepsim":
            return self.encoder(
                self.node_types[rows].long(), self.numeric[rows].float(),
                self.edge_src[rows].long(), self.edge_dst[rows].long()
            ), self.node_types[rows].new_zeros((), dtype=torch.float32)
        return self.encoder(
            self.node_types[rows].long(), self.edge_src[rows].long(), self.edge_dst[rows].long()
        ), self.node_types[rows].new_zeros((), dtype=torch.float32)

    def forward(self, left_rows: Tensor, right_rows: Tensor):
        if self.method == "fa_ast_gmn":
            left, right = self.encoder.forward_pair(
                self.node_types[left_rows].long(), self.edge_src[left_rows].long(), self.edge_dst[left_rows].long(),
                self.node_types[right_rows].long(), self.edge_src[right_rows].long(), self.edge_dst[right_rows].long(),
            )
            auxiliary = left.new_zeros(())
        else:
            left, left_aux = self._single(left_rows)
            right, right_aux = self._single(right_rows)
            auxiliary = left_aux + right_aux
        if self.method == "cdlh":
            logits = 4.0 * (left * right).mean(-1)
        elif self.method == "astnn":
            logits = self.classifier(torch.abs(left - right)).squeeze(-1)
        else:
            features = torch.cat([
                torch.abs(left - right), left * right,
                (left * right).sum(-1, keepdim=True), torch.norm(left - right, dim=-1, keepdim=True)
            ], -1)
            logits = self.classifier(features).squeeze(-1)
        return logits, auxiliary, left, right


def _pack_method(method: str, raw: dict[str, dict], code_ids: list[str], train_ids: set[str]):
    graph_view = "cfg" if method == "deepsim" else "cpg" if method.startswith("fa_ast") else "ast"
    vocab = _training_type_vocabulary(raw, train_ids, graph_view)
    if method in {"astnn", "rtvnn", "cdlh"}:
        names = (
            "node_types", "edge_parents", "edge_children", "depths", "binary_depths",
            "left_child", "right_sibling", "statements"
        )
        rows = [_tree_arrays(raw[code_id]["ast"], vocab)[:8] for code_id in tqdm(code_ids, desc=f"Packing {method} ASTs")]
    elif method == "deepsim":
        names = ("node_types", "numeric", "edge_src", "edge_dst")
        rows = [_deepsim_arrays(raw[code_id], vocab) for code_id in tqdm(code_ids, desc="Packing CFG semantic matrices")]
    else:
        names = ("node_types", "edge_src", "edge_dst")
        rows = [_flow_arrays(raw[code_id], vocab) for code_id in tqdm(code_ids, desc="Packing typed FA-AST relations")]
    arrays = {name: np.stack([row[index] for row in rows]) for index, name in enumerate(names)}
    return arrays, vocab


@torch.no_grad()
def _predict(model, frame, id_to_row, batch_size, device):
    model.eval()
    output = []
    loader = DataLoader(_PairRows(frame, id_to_row), batch_size=batch_size, shuffle=False, num_workers=0)
    for left, right, _ in tqdm(loader, desc="Predict", leave=False):
        logits, _, _, _ = model(left.to(device), right.to(device))
        output.append(torch.sigmoid(logits).float().cpu().numpy())
    return np.concatenate(output) if output else np.empty(0, dtype=np.float32)


def _run_deckard(raw, train, valid, test, id_to_row, train_ids):
    vocab = _training_type_vocabulary(raw, train_ids, "ast")
    vectors = {}
    for code_id in tqdm(id_to_row, desc="DECKARD subtree vectors"):
        types, *_, children = _tree_arrays(raw[code_id]["ast"], vocab)
        n = int((types != 0).sum())
        subtree_vectors, sizes = deckard_subtree_vectors(types[:n], children, len(vocab), min_nodes=3)
        order = np.argsort(sizes)[-32:]
        vectors[code_id] = (subtree_vectors[order], sizes[order])

    def scores(frame):
        return np.asarray([
            deckard_pair_similarity(*vectors[left], *vectors[right])
            for left, right in tqdm(zip(frame.left_id, frame.right_id), total=len(frame), desc="DECKARD pair scoring")
        ], dtype=np.float32)

    valid_scores = scores(valid)
    threshold, valid_metrics = _choose_threshold(valid.label, valid_scores)
    return threshold, valid_metrics, scores(test), 0, [], None


def run_faithful_baseline(dataset_key: str, method: str) -> dict:
    if method not in METHOD_CONFIGS:
        raise ValueError(f"Unknown method {method!r}")
    config = METHOD_CONFIGS[method]
    seed = 42
    _seed_everything(seed)
    started = time.perf_counter()
    bundle_root = _materialize_codenet_zip_input()
    pairs_path = _bundle_file(bundle_root, "pairs.csv.gz", "pairs.csv", "pairs.csv.gz.tmp")
    graphs_path = _bundle_file(bundle_root, "graph_spectra.jsonl.gz", "graph_spectra.jsonl", "graph_spectra.jsonl.gz.tmp")
    with _open_text(pairs_path) as pair_stream:
        pairs = pd.read_csv(pair_stream, dtype={"left_id": str, "right_id": str, "split": str, "label": np.int64})
    train, train_audit = select_nonclone_scope_protocol(pairs[pairs.split.eq("train")], "train")
    valid, valid_audit = select_nonclone_scope_protocol(pairs[pairs.split.eq("valid")], "valid")
    test, test_audit = select_nonclone_scope_protocol(pairs[pairs.split.eq("test")], "test")
    pair_scope_audit = {"train": train_audit, "valid": valid_audit, "test": test_audit}
    code_ids = sorted(set(train.left_id) | set(train.right_id) | set(valid.left_id) | set(valid.right_id) | set(test.left_id) | set(test.right_id))
    train_ids = set(train.left_id) | set(train.right_id)
    raw = _load_raw_graphs(graphs_path, set(code_ids))
    cfg_fallback_ids = []
    if method == "deepsim":
        for code_id in code_ids:
            cfg = raw.get(code_id, {}).get("cfg", {})
            if cfg.get("node_types"):
                continue
            cpg = raw.get(code_id, {}).get("cpg", {})
            if not cpg.get("node_types"):
                raise RuntimeError(f"DeepSim code {code_id} has neither CFG nor CPG adjacency")
            raw[code_id]["cfg"] = cpg
            cfg_fallback_ids.append(code_id)
    required = ("cfg",) if method == "deepsim" else ("ast", "cfg", "ddg", "cpg") if method.startswith("fa_ast") else ("ast",)
    usable = {code_id for code_id in code_ids if all(raw.get(code_id, {}).get(view, {}).get("node_types") for view in required)}
    selected_counts = {"train": len(train), "valid": len(valid), "test": len(test)}
    train = train[train.left_id.isin(usable) & train.right_id.isin(usable)].reset_index(drop=True)
    valid = valid[valid.left_id.isin(usable) & valid.right_id.isin(usable)].reset_index(drop=True)
    test = test[test.left_id.isin(usable) & test.right_id.isin(usable)].reset_index(drop=True)
    retained_counts = {"train": len(train), "valid": len(valid), "test": len(test)}
    if retained_counts != selected_counts:
        raise RuntimeError(f"{method} lost pairs during graph joining: selected={selected_counts}, retained={retained_counts}")
    code_ids = sorted(set(train.left_id) | set(train.right_id) | set(valid.left_id) | set(valid.right_id) | set(test.left_id) | set(test.right_id))
    id_to_row = {code_id: index for index, code_id in enumerate(code_ids)}

    if method == "deckard":
        threshold, valid_metrics, test_scores, parameters, history, best_epoch = _run_deckard(raw, train, valid, test, id_to_row, train_ids)
    else:
        arrays, vocab = _pack_method(method, raw, code_ids, train_ids)
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model = _PairModel(method, arrays, len(vocab)).to(device)
        parameters = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
        optimizer = torch.optim.Adam(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
        scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
        loader = DataLoader(_PairRows(train, id_to_row), batch_size=config["batch"], shuffle=True, num_workers=0, pin_memory=device.type == "cuda")
        history, best = [], None
        patience = RUN_CONFIG["patience"]
        bad_epochs = 0
        for epoch in range(1, RUN_CONFIG["epochs"] + 1):
            model.train()
            total_loss, total_rows = 0.0, 0
            for left, right, labels in tqdm(loader, desc=f"{config['name']} epoch {epoch}"):
                left, right, labels = left.to(device), right.to(device), labels.to(device)
                optimizer.zero_grad(set_to_none=True)
                with torch.amp.autocast("cuda", enabled=device.type == "cuda"):
                    logits, auxiliary, left_code, right_code = model(left, right)
                    if method == "cdlh":
                        loss = CDLHModel.loss(left_code, right_code, labels)
                    else:
                        loss = nn.functional.binary_cross_entropy_with_logits(logits, labels)
                    if method == "rtvnn":
                        loss = loss + 0.05 * auxiliary
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                total_loss += float(loss.detach().cpu()) * len(labels)
                total_rows += len(labels)
            valid_scores = _predict(model, valid, id_to_row, config["batch"], device)
            candidate_threshold, candidate_metrics = _choose_threshold(valid.label, valid_scores)
            row = {"epoch": epoch, "loss": total_loss / max(1, total_rows), **candidate_metrics, "threshold": candidate_threshold}
            history.append(row)
            key = (candidate_metrics["F1"], candidate_metrics["BalancedAccuracy"])
            if best is None or key > best[0]:
                best = (key, copy.deepcopy(model.state_dict()), epoch, candidate_threshold, candidate_metrics)
                bad_epochs = 0
            else:
                bad_epochs += 1
                if bad_epochs >= patience:
                    break
        _, state, best_epoch, threshold, valid_metrics = best
        model.load_state_dict(state)
        test_scores = _predict(model, test, id_to_row, config["batch"], device)

    test_metrics = _metric_dict(test.label, test_scores, threshold)
    runtime = time.perf_counter() - started
    implementation = "paper-faithful adaptation on repository-exported graphs"
    if cfg_fallback_ids:
        implementation += "; complete CPG adjacency substituted only for empty exported CFGs"
    result = {
        "Method": config["name"], "Implementation": implementation,
        "PairScope": "clones_vs_mixed_hard_and_diff_problem_nonclones", "NegativeKinds": ",".join(NONCLONE_KINDS),
        "CfgFallbackCodes": len(cfg_fallback_ids),
        "BestEpoch": best_epoch, "BestValidF1": valid_metrics["F1"], **test_metrics,
        "Threshold": threshold, "TrainPairs": len(train), "ValidPairs": len(valid), "TestPairs": len(test),
        "TrainableParameters": parameters, "RuntimeSeconds": runtime, "RuntimeMinutes": runtime / 60,
        "RunProfile": RUN_PROFILE, "Seed": seed,
    }
    working = Path(globals().get("WORK_DIR_OVERRIDE", "/kaggle/working"))
    working.mkdir(parents=True, exist_ok=True)
    stem = f"{dataset_key}_clone_vs_mixed_aw_diff_{method}_faithful"
    pd.DataFrame([result]).to_csv(working / f"{stem}_results.csv", index=False)
    if history:
        pd.DataFrame(history).to_csv(working / f"{stem}_history.csv", index=False)
    manifest = {
        "generated_at_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "dataset_key": dataset_key, "method": config["name"], "implementation": result["Implementation"],
        "pair_scope": "clones_vs_mixed_hard_and_diff_problem_nonclones", "negative_kinds": list(NONCLONE_KINDS),
        "pair_scope_audit": pair_scope_audit, "cfg_fallback_code_ids": cfg_fallback_ids,
        "input_graphs": list(required), "self_pair_count": 0, "hyperparameters": config,
        "profile": RUN_PROFILE, "configured_epochs": RUN_CONFIG["epochs"], "seed": seed,
        "selected_epoch": best_epoch, "selected_threshold": threshold, "result": result,
    }
    (working / f"{stem}_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    record_language_breakdown(test, test_scores, threshold, dataset=f"{dataset_key}_clone_vs_mixed_aw_diff", method=config["name"])
    print(pd.DataFrame([result]))
    return result

FAITHFUL_RESULTS = [run_faithful_baseline(key, 'astnn') for key in DATASET_KEYS]
